In [1]:
from config_008b import *
from parsing_utils_main_tissue import *
from plotting_utils_main_tissue import *
from pdf_report_multi_tissue import *

import pandas as pd
import os


<frozen importlib._bootstrap>:219: RuntimeWarning: scipy._lib.messagestream.MessageStream size changed, may indicate binary incompatibility. Expected 56 from C header, got 64 from PyObject


In [2]:
# Load data
sample_df_full = parse_sample_df(sample_metadata_path)
config_df = pd.read_csv(config_path, sep='\t')
sample_df_this_plate = sample_df_full[sample_df_full['plate'] == plate]

# Get main tissue
main_tisss = sample_df_this_plate[sample_df_this_plate['well_type'] == 'Single']['Tissue'].unique().tolist()
main_tiss = main_tisss[0]
main_tiss2 = main_tisss[1]
multi_tiss = sample_df_this_plate[sample_df_this_plate['well_type'] == 'Multiplexed']['Tissue'].iloc[0]


sample_df_this_tissue = sample_df_full[sample_df_full['Tissue'] == multi_tiss]
sample_df_this_tissue =sample_df_this_tissue[sample_df_this_tissue['plate'].isin(['igvf_003', 'igvf_004', 'igvf_005', 'igvf_007', 'igvf_008', 'igvf_008b', 'igvf_009', 'igvf_010', 'igvf_011', 'igvf_012'])]
sample_df_this_tissue = sample_df_this_tissue[sample_df_this_tissue['plate'] != plate]
all_plates_list  = sample_df_this_tissue['plate'].unique().tolist()



In [3]:
sample_df = sample_df_this_plate[sample_df_this_plate['Tissue'] == multi_tiss]
color_by = 'Tissue' if sample_df_this_plate['Tissue'].nunique() > sample_df['Genotype'].nunique() else 'genotype'
color_dict = get_color_dict(sample_df_this_plate, color_by)
# Plot plate map
plate_map_path = plot_plate_map(sample_df_this_plate, color_by, color_dict)

# Subpool setup
subpools = get_subpools(config_df, plate)

# Get main and multiplexed tissues
combined_obs = load_combined_obs(sample_df)


# Alignment stats
align_df = load_alignment_stats(plate, subpools)

# Load obs table
adata_obs = pd.read_csv(obs_path)
adata_obs =adata_obs[adata_obs['Tissue'] == multi_tiss]


# QC summary tables
qc_200umi, qc_500umi = load_qc_stats(adata_obs, multi_tiss, subpools, sampletype)

# Cell counts
formatted_counts = summarize_cell_counts(qc_200umi, qc_500umi, sampletype)

# Generate knee plot
knee_plot_path = plot_knee_raw_counts(plate, subpools)


# Heatmaps
hmap_paths = [create_round_heatmap(adata_obs, round_col, kit, sampletype) for round_col in ['bc1_well', 'bc2_well', 'bc3_well']]

# Cellbender
cb_results_df = load_cellbender_stats(plate, subpools, sampletype)
cb_settings_df = load_cellbender_settings(plate, subpools)
cb_knee_path = plot_knee_cb(adata_obs)

# Violin plots
violin1_path, violin2_path = plot_qc_violins(adata_obs)
adata_obs_filt = filter_obs(adata_obs, min_counts, max_counts, min_genes, pct_counts_mt, doublet_score)
violin_filt1, violin_filt2 = plot_qc_violins_filtered(adata_obs_filt)

mult_celltype_path = plot_stacked_mult(combined_obs, multi_tiss, "plots/sample_celltype_proportions_main.png")

# Barcode map
barcode_map_df = create_barcode_sample_map(plate, kit, chemistry)
barcode_map_df = barcode_map_df[barcode_map_df['well'].str.contains(r'(9|10|11|12)$')] # YES multiplexed wells


/share/crsp/lab/seyedam/share/8cube_paper/plate_report/split_by_tissue/parsing_utils_main_tissue.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['genotype'] = df['Genotype']
/share/crsp/lab/seyedam/share/8cube_paper/plate_report/split_by_tissue/parsing_utils_main_tissue.py:66: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['genotype'] = df.apply(assign_label, axis=1)
/share/crsp/lab/seyedam/share/8cube_paper/plate_report/split_by_tissue/plotting_utils_main_tissue.py:14: SettingWithCopyWarning: 
A 

In [4]:
# Create report
elements = build_pdf_report(
    plate,
    config_df,
    kit,
    chemistry,
    sample_df,
    subpools,
    color_by,
    plate_map_path,
    align_df,
    formatted_counts,
    qc_200umi,
    qc_500umi,
    knee_plot_path,
    hmap_paths,
    cb_settings_df,
    cb_results_df,
    cb_knee_path, 
    violin1_path,
    violin2_path,
    violin_filt1,
    violin_filt2,
    mult_celltype_path,
    barcode_map_df,
    adata_obs,
    adata_obs_filt,
    combined_obs,
    main_tiss,
    multi_tiss,
    sampletype,
    min_counts, 
    min_genes, 
    max_counts, 
    pct_counts_mt, 
    doublet_score,
    all_plates_list,
    main_tissue2 = main_tiss2
    
    
)




In [5]:
# Save PDF
doc = create_pdf_doc(plate, multi_tiss)
doc.build(elements)
print(f"Report saved as {doc.filename}")


Report saved as igvf_008b/igvf_008b_report_Adrenal.pdf
